In [ ]:
# Celda 1
# Openings with depth ("derrame"). Every opening (glass, dark arch, door, niche) is cut out of the wall behind it and of
# the stone plates around it (frames, ashlar plates), and set back by the depth of its reveal; the sides of the hole
# (jambs, sill, lintel, soffit of the arch) are new faces. Runs after 15_volumenes and before 14_materiales.
%cd ..

IN_JSON = "data/lod3_volumenes_meshes.json"     # 15_volumenes
JSON_FILE = "data/lod3_detalles_meshes.json"    # Read by 14_materiales
REPORT = "data/control/derrames.csv"            # One row per opening; read by 16_control
GPKG = "data/catastro_manzana.gpkg"
VOLUMES_CSV = "data/volumenes/volumenes.csv"
ORIGIN = (440000.0, 4473700.0, 640.0)
INSTITUTO_PARCELS = ["0139424VK4703G", "0139423VK4703G"]

# Depth of the reveal (m) behind the wall face, by name: the first pattern that matches wins. [ESTIMADO] from the
# shadows on the photos (window frames about a quarter of a meter back on photo 01; deep porch of the Colegiata and
# dark gallery behind the cloister arches on the 2022 photo and on a photo of the cloister of unknown license, used only as a reference).
DEPTHS = [
    (r"arco_claustro", 0.9),      # Piers of the cloister arcade
    (r"arco_portada", 1.5),       # Porch of the Colegiata
    (r"hornacina", 0.6),          # Niche with the statues
    (r"puerta", 0.4),             # Doors of the portals of the instituto
    (r"torre_|campanario", 0.6),  # Openings of the towers
    (r"cupula|linterna", 0.35),   # Windows of the drum, attic and lantern
    (r".", 0.25),                 # Windows
]
# The plate at the back of the hole keeps the material of the opening, except:
BACK_MATERIAL = [(r"puerta|arco_portada", "madera"), (r"hornacina", "granito_liso")]
HOST_GAP = 0.7        # The wall behind an opening is at most this far from its glass (m)
PARALLEL = 0.99       # |cos| of the angle between an opening and its wall (estimated windows follow a wall up to 6 degrees off)
LEVEL_TOL = 0.01      # A plate is a prism along the wall normal if all its vertices are on two planes (m)

In [ ]:
# Celda 2
import os
import re
import csv
import json
import math
import numpy as np
import shapely
import geopandas as gpd
from shapely.geometry import Polygon, Point, MultiPoint
from shapely.geometry.polygon import orient

with open(IN_JSON, encoding="utf-8") as f:
    meshes = json.load(f)["parts"]
with open(VOLUMES_CSV, encoding="utf-8") as f:
    PIECES = {row["pieza"] for row in csv.DictReader(f)}
block = shapely.union_all([shapely.affinity.translate(g, -ORIGIN[0], -ORIGIN[1]) for g in gpd.read_file(GPKG, layer="building_parts").geometry])

def first_match(rules, name):
    return next(value for pattern, value in rules if re.search(pattern, name))

def newell(points):
    p = np.asarray(points, float)
    q = np.roll(p, -1, axis=0)
    n = np.array([np.sum((p[:, 1] - q[:, 1]) * (p[:, 2] + q[:, 2])), np.sum((p[:, 2] - q[:, 2]) * (p[:, 0] + q[:, 0])),
                  np.sum((p[:, 0] - q[:, 0]) * (p[:, 1] + q[:, 1]))])
    area = np.linalg.norm(n) / 2
    return (n / (2 * area) if area > 0 else n), area

def is_opening(m):
    if m.get("lod") != 3 or m.get("group") == "metal" or m.get("material_hint") == "hierro":
        return False
    return m.get("material_hint") in ("ventana", "sombra") or m.get("group") == "vidrio"

def plate_levels(vertices, out):
    """The two levels along `out` of a prism drawn along the wall normal, or None if it is not one."""
    d = np.asarray(vertices) @ np.array([out[0], out[1], 0.0])
    lo, hi = d.min(), d.max()
    if hi - lo < 0.003 or not np.all((np.abs(d - lo) < LEVEL_TOL) | (np.abs(d - hi) < LEVEL_TOL)):
        return None
    return lo, hi

# Wall faces that can hold an opening: vertical faces of the LOD2 parts of the ensemble and of the pieces of 15.
host_index = []
for mi, m in enumerate(meshes):
    if not ((m.get("lod") == 2 and m.get("group") == "conjunto") or m["name"] in PIECES):
        continue
    for fi, face in enumerate(m["faces"]):
        points = [m["vertices"][i] for i in face]
        normal, area = newell(points)
        if area < 0.05 or abs(normal[2]) > 0.1:   # Slightly battered faces too (attic of the dome)
            continue
        horizontal = normal[:2] / np.linalg.norm(normal[:2])
        p = np.asarray(points)
        host_index.append((mi, fi, horizontal[0], horizontal[1], float(p[0, :2] @ horizontal), p[:, 0].min(), p[:, 0].max(),
                           p[:, 1].min(), p[:, 1].max(), p[:, 2].min(), p[:, 2].max(), p[0, 0], p[0, 1]))
H = np.array([h[2:] for h in host_index])
openings = [i for i, m in enumerate(meshes) if is_opening(m)]
print(len(openings), "openings |", len(host_index), "vertical faces that can hold them")

In [ ]:
# Celda 3
# For each opening: its frame (center c, outward normal "out", t along the wall), its hole (outline in (s, z)), the
# wall faces behind it, and the stone plates around it that must be cut.
def frame_of(m):
    best = max((newell([m["vertices"][i] for i in face]) for face in m["faces"]), key=lambda na: na[1] if abs(na[0][2]) < 0.1 else -1)
    normal = best[0][:2] / np.linalg.norm(best[0][:2])
    return np.mean(np.asarray(m["vertices"])[:, :2], axis=0), normal

def to_sz(points, c, t):
    p = np.asarray(points, float)
    return np.c_[(p[:, :2] - c) @ t, p[:, 2]]

# Plates: LOD3 elements that are not openings, iron or pieces of 15 (a square tower body is also a prism along its
# normal, but it holds openings, it is not a plate on a wall).
lod3_boxes = [(i, np.asarray(m["vertices"])) for i, m in enumerate(meshes) if m.get("lod") == 3 and not is_opening(m)
              and m.get("group") != "metal" and m.get("material_hint") != "hierro" and m["name"] not in PIECES]
lod3_bounds = np.array([[v[:, 0].min(), v[:, 0].max(), v[:, 1].min(), v[:, 1].max(), v[:, 2].min(), v[:, 2].max()] for _, v in lod3_boxes])

wall_holes = {}      # (mesh, face) -> list of holes, each a list of 3D points
plate_holes = {}     # mesh -> (frame (c, t, out), list of holes in that frame)
plans = []           # What to do with each opening
hidden = set()       # Estimated windows inside other volumes, removed
report = []
for oi in openings:
    m = meshes[oi]
    c, normal = frame_of(m)
    v = np.asarray(m["vertices"])
    d = (v[:, :2] - c) @ normal
    # Wall faces parallel to the opening, near its plane, overlapping it:
    cos = H[:, 0] * normal[0] + H[:, 1] * normal[1]
    # Where the plane of the face crosses the line through c along `normal` (exact also for walls a bit off parallel):
    position = ((H[:, 9] - c[0]) * H[:, 0] + (H[:, 10] - c[1]) * H[:, 1]) / np.where(np.abs(cos) > 1e-6, cos, 1e-6)
    near = (np.abs(cos) > PARALLEL) & (position > d.min() - HOST_GAP) & (position < d.max() + HOST_GAP)
    near &= (H[:, 3] < v[:, 0].max() + 0.1) & (H[:, 4] > v[:, 0].min() - 0.1) & (H[:, 5] < v[:, 1].max() + 0.1) & (H[:, 6] > v[:, 1].min() - 0.1)
    near &= (H[:, 7] < v[:, 2].max()) & (H[:, 8] > v[:, 2].min())
    t0 = np.array([-normal[1], normal[0]])
    hole0 = MultiPoint(to_sz(v, c, t0)).convex_hull
    candidates = []
    for k in np.flatnonzero(near):
        mi, fi = host_index[k][:2]
        face_poly = Polygon(to_sz([meshes[mi]["vertices"][i] for i in meshes[mi]["faces"][fi]], c, t0))
        if not face_poly.is_valid:
            face_poly = face_poly.buffer(0)
        overlap = face_poly.intersection(hole0).area
        if overlap > 0.02 * hole0.area:
            candidates.append((k, overlap, position[k]))
    name = m["name"]
    if not candidates:
        if name.startswith("hueco_estimado"):
            # An estimated window (15, cell 5d) on a bay wall that another bay covers: it is inside a volume, unseen.
            number = int(name.split("_")[2])
            hidden.update({name, f"hueco_estimado_{number - 1}_marco"})
            report.append({"hueco": name, "muro": "", "fondo_m": "", "estado": "eliminado: dentro de un volumen"})
        else:
            report.append({"hueco": name, "muro": "", "fondo_m": "", "estado": "sin muro detrás"})
        continue
    # The wall is the plane with most overlap; every face on that plane (a wall cut in pieces) gets the hole.
    main = max(candidates, key=lambda x: x[1])
    hosts = [k for k, _, pos in candidates if abs(pos - main[2]) < 0.05]
    host_mesh = meshes[host_index[main[0]][0]]
    # Outward: the side that is outside the block (streets, courtyards); if both or none, away from the host.
    plus, minus = (block.contains(Point(*(c + s * 0.6 * normal))) for s in (1, -1))
    if host_mesh["name"] in PIECES or plus == minus:
        centroid = np.asarray(host_mesh["vertices"])[:, :2].mean(axis=0)
        out = normal if (c - centroid) @ normal > 0 else -normal
    else:
        out = -normal if plus else normal
    t = np.array([-out[1], out[0]])
    n_glass = (v[:, :2] - c) @ out
    g1, g2 = n_glass.min(), n_glass.max()
    n_wall = main[2] * (1 if out @ normal > 0 else -1)
    hole = MultiPoint(to_sz(v, c, t)).convex_hull
    depth = first_match(DEPTHS, name)
    n_back = min(n_wall, g1) - depth
    for k in hosts:
        wall_holes.setdefault(host_index[k][:2], []).append([(*(c + s * t), z) for s, z in hole.exterior.coords[:-1]])
    # Stone plates around it: prisms along `out` that reach from the wall to the glass and overlap the hole.
    x0, x1, y0, y1, z0, z1 = v[:, 0].min(), v[:, 0].max(), v[:, 1].min(), v[:, 1].max(), v[:, 2].min(), v[:, 2].max()
    close = np.flatnonzero((lod3_bounds[:, 0] < x1 + 1) & (lod3_bounds[:, 1] > x0 - 1) & (lod3_bounds[:, 2] < y1 + 1) &
                           (lod3_bounds[:, 3] > y0 - 1) & (lod3_bounds[:, 4] < z1) & (lod3_bounds[:, 5] > z0))
    front = n_wall
    stone = False
    for k in close:
        pi, pv = lod3_boxes[k]
        levels = plate_levels(pv, out)
        if levels is None:
            continue
        p1, p2 = levels[0] - c @ out, levels[1] - c @ out
        if p1 > g2 + 0.005 or p2 < n_wall - 0.05:
            continue
        shape = MultiPoint(to_sz(pv, c, t)).convex_hull
        if shape.intersection(hole).area < 0.01 * hole.area:
            continue
        if pi not in plate_holes:
            plate_holes[pi] = ((c, t, out), [])
        pc, pt, _ = plate_holes[pi][0]
        plate_holes[pi][1].append([((c + s * t - pc) @ pt, z) for s, z in hole.exterior.coords[:-1]])
        front = max(front, p2)
        stone |= meshes[pi].get("group") == "piedra" or str(meshes[pi].get("material_hint", "")).startswith("granito")
    if stone:
        reveal_material = "granito_liso"
    elif host_mesh["name"] in PIECES:
        reveal_material = host_mesh["material_hint"]
    else:
        reveal_material = "muro_instituto" if host_mesh.get("parcel") in INSTITUTO_PARCELS else "muro_colegiata"
    plans.append({"index": oi, "c": c, "t": t, "out": out, "hole": hole, "front": front, "back": n_back, "g1": g1,
                  "material": reveal_material, "parcel": host_mesh.get("parcel", "")})
    report.append({"hueco": name, "muro": host_mesh["name"], "fondo_m": round(front - n_back, 2), "estado": "OK"})
print(len(plans), "openings set back |", len(openings) - len(plans), "without a wall behind |",
      len(wall_holes), "wall faces and", len(plate_holes), "plates to cut")

In [ ]:
# Celda 4
# Cut: wall faces become triangles around their holes (in a new mesh "<part>_huecos", so that 14_materiales still
# reads them as walls), plates are rebuilt as prisms with holes, the reveals are added and the openings moved back.
def triangles_3d(polygon, to_xyz):
    result = []
    for poly in getattr(polygon, "geoms", [polygon]):
        if poly.geom_type != "Polygon" or poly.area < 1e-6:
            continue
        for tri in shapely.constrained_delaunay_triangles(orient(poly, 1.0)).geoms:
            result.append([to_xyz(s, z) for s, z in list(orient(tri, 1.0).exterior.coords)[:3]])
    return result

by_mesh = {}
for (mi, fi), holes in wall_holes.items():
    by_mesh.setdefault(mi, {})[fi] = holes
new_meshes = []
for mi, faces in by_mesh.items():
    m = meshes[mi]
    vertices, new_faces = [], []
    for fi, holes in faces.items():
        points = np.asarray([m["vertices"][i] for i in m["faces"][fi]])
        normal, _ = newell(points)
        t = np.array([-normal[1], normal[0]]) / np.linalg.norm(normal[:2])
        p0 = points[0, :2]
        face_poly = Polygon(np.c_[(points[:, :2] - p0) @ t, points[:, 2]]).buffer(0)
        cut = face_poly.difference(shapely.union_all([Polygon(np.c_[(np.asarray(h)[:, :2] - p0) @ t, np.asarray(h)[:, 2]]) for h in holes]))
        for tri in triangles_3d(cut, lambda s, z: (*(p0 + s * t), z)):
            new_faces.append(tuple(range(len(vertices), len(vertices) + 3)))
            vertices += [tuple(map(float, p)) for p in tri]
    m["faces"] = [f for k, f in enumerate(m["faces"]) if k not in faces]
    new_meshes.append({**{k: m[k] for k in m if k not in ("vertices", "faces")}, "name": m["name"] + "_huecos",
                       "vertices": vertices, "faces": new_faces})

for pi, ((c, t, out), holes) in plate_holes.items():
    m = meshes[pi]
    v = np.asarray(m["vertices"])
    levels = plate_levels(v, out)
    p1, p2 = levels[0] - c @ out, levels[1] - c @ out
    # Shape of the plate: union of its faces on the front level (it may already have holes, like an eared frame).
    caps = []
    for face in m["faces"]:
        fv = v[list(face)]
        if np.all(np.abs((fv[:, :2] - c) @ out - p2) < LEVEL_TOL):
            poly = Polygon(np.c_[(fv[:, :2] - c) @ t, fv[:, 2]])
            caps.append(poly if poly.is_valid else poly.buffer(0))
    shape = shapely.union_all(caps).difference(shapely.union_all([Polygon(h) for h in holes]))
    vertices, faces = [], []
    def add(s, n, z):
        vertices.append((*(c + s * t + n * out), z))
        return len(vertices) - 1
    for poly in getattr(shape, "geoms", [shape]):
        if poly.geom_type != "Polygon" or poly.area < 1e-6:
            continue
        poly = orient(poly, 1.0)
        for tri in shapely.constrained_delaunay_triangles(poly).geoms:
            pts = list(orient(tri, 1.0).exterior.coords)[:3]
            faces.append(tuple(add(s, p1, z) for s, z in reversed(pts)))
            faces.append(tuple(add(s, p2, z) for s, z in pts))
        for ring_line in [poly.exterior, *poly.interiors]:
            r = list(ring_line.coords)[:-1]
            for (sa, za), (sb, zb) in zip(r, r[1:] + r[:1]):
                faces.append((add(sa, p1, za), add(sb, p1, zb), add(sb, p2, zb), add(sa, p2, za)))
    m["vertices"] = [tuple(map(float, p)) for p in vertices]
    m["faces"] = faces

for plan in plans:
    m = meshes[plan["index"]]
    c, t, out = plan["c"], plan["t"], plan["out"]
    ring_sz = list(plan["hole"].exterior.coords)[:-1]
    vertices, faces = [], []
    for s, z in ring_sz:
        for n in (plan["front"], plan["back"]):
            vertices.append((*map(float, c + s * t + n * out), float(z)))
    for k in range(len(ring_sz)):
        j = (k + 1) % len(ring_sz)
        faces.append((2 * k, 2 * j, 2 * j + 1, 2 * k + 1))
    new_meshes.append({"name": "derrame_" + m["name"], "parcel": plan["parcel"], "group": "volumen", "material_hint": plan["material"],
                       "lod": 3, "height_m": 0.0, "height_method": "[ESTIMADO] fondo del hueco: " + str(first_match(DEPTHS, m["name"])) + " m",
                       "vertices": vertices, "faces": faces})
    shift = (plan["back"] - plan["g1"]) * out
    m["vertices"] = [(x + float(shift[0]), y + float(shift[1]), z) for x, y, z in m["vertices"]]
    back = next((value for pattern, value in BACK_MATERIAL if re.search(pattern, m["name"])), None)
    if back:
        m["material_hint"] = back

result = [m for m in meshes if m["faces"] and m["name"] not in hidden] + new_meshes
os.makedirs(os.path.dirname(REPORT), exist_ok=True)
with open(REPORT, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["hueco", "muro", "fondo_m", "estado"])
    writer.writeheader()
    writer.writerows(report)
with open(JSON_FILE, "w", encoding="utf-8") as f:
    json.dump({"origin": ORIGIN, "parts": result}, f)
print(len(result), "meshes |", sum(1 for r in report if r["estado"] == "sin muro detrás"), "openings without a wall behind |",
      len(hidden) // 2, "hidden estimated windows removed (see", REPORT + ")")